## Brute force research into pyzeo
## Tutorial is more interesting, this is just inspecting and creating the first incomplete graph

most cells were removed that were not interesting, only (old) full solutions are left.

In [10]:
import sys, tempfile, os
sys.path.insert(0, '/home/milan/Downloads/alignn')
sys.path.insert(0, '/home/milan/Downloads/alignn/Thesis/Pore_plan_b')

from jarvis.db.figshare import data as jdata
from jarvis.core.atoms import Atoms
from pyzeo.netstorage import AtomNetwork

raw = jdata('hmof')
atoms = Atoms.from_dict(raw[0]['atoms'])

# Write CSSR (the working format)
def atoms_to_cssr(atoms, path):
    lat = atoms.lattice_mat
    elements = atoms.elements
    coords = atoms.frac_coords
    a, b, c = lat[0][0], lat[1][1], lat[2][2]
    with open(path, 'w') as f:
        f.write(f"                {a:.4f}  {b:.4f}  {c:.4f}\n")
        f.write(f"   90.0000   90.0000   90.0000   SPGR =  1 P 1\n")
        f.write(f" {len(elements)} 0\n")
        f.write(f" 0 {atoms.composition.formula}\n")
        for i, (el, fc) in enumerate(zip(elements, coords)):
            f.write(f"{i+1} {el} {fc[0]:.6f} {fc[1]:.6f} {fc[2]:.6f}  0  0  0  0  0  0  0  0 0.000\n")

with tempfile.NamedTemporaryFile(suffix=".cssr", delete=False, mode='w') as tmp:
    cssr_path = tmp.name
atoms_to_cssr(atoms, cssr_path)

# Get the VoronoiNetwork object and inspect it
atm_ntw = AtomNetwork.read_from_CSSR(cssr_path.encode(), rad_flag=False)
vornet, bef, aft = atm_ntw.perform_voronoi_decomposition()

print("VoronoiNetwork type:", type(vornet))
print("VoronoiNetwork attributes:", [a for a in dir(vornet) if not a.startswith('_')])
print()

# Also inspect bef and aft
print("bef type:", type(bef))
print("bef attributes:", [a for a in dir(bef) if not a.startswith('_')])
print()
print("aft type:", type(aft))
print("aft attributes:", [a for a in dir(aft) if not a.startswith('_')])

os.unlink(cssr_path)

Obtaining hMOF dataset 137k...
Reference:https://doi.org/10.1021/acs.jpcc.6b08729
Loading the zipfile...
Loading completed.
False None
Reading input file: /tmp/tmpw5z1ndgp.cssr
VoronoiNetwork type: <class 'pyzeo.extension.VoronoiNetwork'>
VoronoiNetwork attributes: ['analyze_writeto_XYZ', 'perform_voronoi_decomposition', 'prune', 'size', 'write_to_XYZ']

bef type: <class 'list'>
bef attributes: ['append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']

aft type: <class 'list'>
aft attributes: ['append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']
Box dimensions:
  va=(19.077100 0 0)
  vb=(0.000000 21.378400 0)
  vc=(0.000000 0.000000 21.377500)

Total particles = 123

Internal grid size = (3 3 3)

Using voro++ without radii for particles.
Performing Voronoi decomposition.
Volume check:
  Total domain volume  = 8718.554165
  Total Voronoi volume = 8718.554165
Voronoi decomposition finished. Rerout

In [ ]:
import sys, tempfile, os, pyzeo
sys.path.insert(0, '/home/milan/Downloads/alignn')
sys.path.insert(0, '/home/milan/Downloads/alignn/Thesis/Pore_plan_b')

from jarvis.db.figshare import data as jdata
from jarvis.core.atoms import Atoms
from pyzeo.netstorage import AtomNetwork

# Find pyzeo's built-in radii file
pyzeo_dir = os.path.dirname(pyzeo.__file__)
print("pyzeo directory:", pyzeo_dir)
print("pyzeo files:")
for f in os.listdir(pyzeo_dir):
    print(" ", f)

# Also check one level up
print("\nFiles in pyzeo parent:")
parent = os.path.dirname(pyzeo_dir)
for f in os.listdir(parent):
    if 'rad' in f.lower() or 'zeo' in f.lower():
        print(" ", f)

# Search for any .rad file
import subprocess
result = subprocess.run(['find', os.path.dirname(pyzeo_dir), '-name', '*.rad'], 
                      capture_output=True, text=True)
print("\n.rad files found:")
print(result.stdout)

raw = jdata('hmof')
atoms = Atoms.from_dict(raw[0]['atoms'])

def atoms_to_cssr(atoms, path):
    lat = atoms.lattice_mat
    elements = atoms.elements
    coords = atoms.frac_coords
    a, b, c = lat[0][0], lat[1][1], lat[2][2]
    with open(path, 'w') as f:
        f.write(f"                {a:.4f}  {b:.4f}  {c:.4f}\n")
        f.write(f"   90.0000   90.0000   90.0000   SPGR =  1 P 1\n")
        f.write(f" {len(elements)} 0\n")
        f.write(f" 0 {atoms.composition.formula}\n")
        for i, (el, fc) in enumerate(zip(elements, coords)):
            f.write(f"{i+1} {el} {fc[0]:.6f} {fc[1]:.6f} {fc[2]:.6f}  0  0  0  0  0  0  0  0 0.000\n")

with tempfile.NamedTemporaryFile(suffix=".cssr", delete=False, mode='w') as tmp:
    cssr_path = tmp.name
atoms_to_cssr(atoms, cssr_path)

# Write our own radii file, standard atomic radii for elements in hMOF
# (Zn, C, H, N, O are the main ones for hMOF-3476)
radii = {
    'H': 1.20, 'C': 1.70, 'N': 1.55, 'O': 1.52,
    'Zn': 1.39, 'Cu': 1.40, 'Co': 1.26, 'Ni': 1.24,
    'Fe': 1.26, 'Mn': 1.61, 'Cr': 1.66, 'Al': 1.84,
    'Si': 2.10, 'S': 1.80, 'F': 1.47, 'Cl': 1.75,
    'Br': 1.85, 'I': 1.98, 'P': 1.80, 'B': 1.92,
}

rad_path = cssr_path.replace('.cssr', '.rad')
with open(rad_path, 'w') as f:
    for el, r in radii.items():
        f.write(f"{el} {r}\n")
print("Radii file written:", rad_path)

# Now try with rad_flag=True and our radii file
print("\nTrying read_from_CSSR with rad_flag=True and radii file...")
try:
    atm_ntw = AtomNetwork.read_from_CSSR(
        cssr_path.encode(), rad_flag=True, rad_file=rad_path.encode()
    )
    vornet, bef, aft = atm_ntw.perform_voronoi_decomposition()
    print(f"bef nodes: {len(bef)}")
    print(f"aft nodes: {len(aft)}")
    print(f"vornet size (edges): {vornet.size()}")
    print(f"First bef node: {bef[0]}")

    # Try free sphere parameters now that radii are set
    result = atm_ntw.calculate_free_sphere_parameters(cssr_path.encode())
    print(f"free sphere params: {result}")

    # Try write_to_XYZ with floats
    xyz_path = cssr_path.replace('.cssr', '.xyz')
    vornet.write_to_XYZ(cssr_path.encode(), xyz_path.encode(), 0.0, 1.2)
    with open(xyz_path) as f:
        lines = f.readlines()
    print(f"\nXYZ lines: {len(lines)}")
    print("First 15 lines:")
    print("".join(lines[:15]))
    os.unlink(xyz_path)

except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")

os.unlink(cssr_path)
os.unlink(rad_path)

pyzeo directory: /home/milan/miniconda3/envs/mof/lib/python3.10/site-packages/pyzeo
pyzeo files:
  cycle
  high_accuracy
  extension.cpp
  __pycache__
  extension.cpython-310-x86_64-linux-gnu.so
  netstorage
  cluster
  psd
  __init__.py
  area_volume

Files in pyzeo parent:
  pyzeo-0.1.7.dist-info
  pyzeo

.rad files found:

Obtaining hMOF dataset 137k...
Reference:https://doi.org/10.1021/acs.jpcc.6b08729
Loading the zipfile...
Loading completed.
Radii file written: /tmp/tmpo0_cnqvf.rad

Trying read_from_CSSR with rad_flag=True and radii file...
True b'/tmp/tmpo0_cnqvf.rad'
Reading input file: /tmp/tmpo0_cnqvf.cssr
bef nodes: 1511
aft nodes: 904
vornet size (edges): 746
First bef node: (17.16726019733304, 11.112597246807237, 5.983940676222283)
free sphere params: None
FAILED: TypeError: write_to_XYZ() takes at most 2 positional arguments (4 given)
Box dimensions:
  va=(19.077100 0 0)
  vb=(0.000000 21.378400 0)
  vc=(0.000000 0.000000 21.377500)

Total particles = 123

Internal grid s

In [ ]:
import sys, tempfile, os, pyzeo
import numpy as np
sys.path.insert(0, '/home/milan/Downloads/alignn')
sys.path.insert(0, '/home/milan/Downloads/alignn/Thesis/Pore_plan_b')

from jarvis.db.figshare import data as jdata
from jarvis.core.atoms import Atoms
from pyzeo.netstorage import AtomNetwork

raw = jdata('hmof')
atoms = Atoms.from_dict(raw[0]['atoms'])

RADII = {
    'H': 1.20, 'C': 1.70, 'N': 1.55, 'O': 1.52,
    'Zn': 1.39, 'Cu': 1.40, 'Co': 1.26, 'Ni': 1.24,
    'Fe': 1.26, 'Mn': 1.61, 'Cr': 1.66, 'Al': 1.84,
    'Si': 2.10, 'S': 1.80, 'F': 1.47, 'Cl': 1.75,
    'Br': 1.85, 'I': 1.98, 'P': 1.80, 'B': 1.92,
}

def atoms_to_cssr(atoms, path):
    lat = atoms.lattice_mat
    elements = atoms.elements
    coords = atoms.frac_coords
    a, b, c = lat[0][0], lat[1][1], lat[2][2]
    with open(path, 'w') as f:
        f.write(f"                {a:.4f}  {b:.4f}  {c:.4f}\n")
        f.write(f"   90.0000   90.0000   90.0000   SPGR =  1 P 1\n")
        f.write(f" {len(elements)} 0\n")
        f.write(f" 0 {atoms.composition.formula}\n")
        for i, (el, fc) in enumerate(zip(elements, coords)):
            f.write(f"{i+1} {el} {fc[0]:.6f} {fc[1]:.6f} {fc[2]:.6f}  0  0  0  0  0  0  0  0 0.000\n")

def write_rad_file(atoms, path):
    with open(path, 'w') as f:
        for el in set(atoms.elements):
            f.write(f"{el} {RADII.get(el, 1.5)}\n")

with tempfile.NamedTemporaryFile(suffix=".cssr", delete=False, mode='w') as tmp:
    cssr_path = tmp.name
rad_path = cssr_path.replace('.cssr', '.rad')
atoms_to_cssr(atoms, cssr_path)
write_rad_file(atoms, rad_path)

atm_ntw = AtomNetwork.read_from_CSSR(cssr_path.encode(), rad_flag=True, rad_file=rad_path.encode())
vornet, bef, aft = atm_ntw.perform_voronoi_decomposition()

# Try analyze_writeto_XYZ with correct signature
print("=== analyze_writeto_XYZ ===")
axyz_path = cssr_path.replace('.cssr', '_analyze.xyz')
try:
    vornet.analyze_writeto_XYZ(axyz_path.encode(), 1.2, atm_ntw)
    with open(axyz_path) as f:
        lines = f.readlines()
    print(f"Lines: {len(lines)}")
    print("First 25 lines:")
    print("".join(lines[:25]))
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")

# write_to_XYZ with floats
print("\n=== write_to_XYZ with floats ===")
xyz_path = cssr_path.replace('.cssr', '.xyz')
try:
    vornet.write_to_XYZ(xyz_path, 1.2)   # string path, float probe radius
    with open(xyz_path) as f:
        lines = f.readlines()
    print(f"Lines: {len(lines)}")
    print("First 25 lines:")
    print("".join(lines[:25]))
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")

# Try vornet.prune(), may return accessible subnetwork
print("\n=== vornet.prune() ===")
try:
    import inspect
    print("prune signature:", inspect.signature(vornet.prune))
    pruned = vornet.prune(1.2)
    print(f"pruned type: {type(pruned)}")
    print(f"pruned attrs: {[a for a in dir(pruned) if not a.startswith('_')]}")
    if pruned is not None:
        print(f"pruned size: {pruned.size()}")
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")

# Build pore graph manually from bef nodes
# We have node positions, compute radii manually, connect nearby nodes
print("\n=== Manual pore graph construction ===")
atom_positions = np.array(atoms.cart_coords)
pore_positions = np.array(bef)
probe_radius = 1.2

# Compute pore radius for every node
pore_radii = []
for pos in pore_positions:
    dists = np.linalg.norm(atom_positions - pos, axis=1)
    pore_radii.append(dists.min())
pore_radii = np.array(pore_radii)

# Filter to accessible nodes
accessible_mask = pore_radii > probe_radius
accessible_positions = pore_positions[accessible_mask]
accessible_radii = pore_radii[accessible_mask]

print(f"Total Voronoi nodes: {len(pore_positions)}")
print(f"Accessible nodes (radius > {probe_radius}A): {accessible_mask.sum()}")
print(f"Pore radius stats: min={accessible_radii.min():.2f} "
      f"max={accessible_radii.max():.2f} mean={accessible_radii.mean():.2f}")

# Connect nodes within cutoff distance (edge = two pores are adjacent)
cutoff = 8.0   # Angstroms — pores within this distance get an edge
from scipy.spatial.distance import cdist
dist_matrix = cdist(accessible_positions, accessible_positions)
edge_src, edge_dst = np.where((dist_matrix < cutoff) & (dist_matrix > 0))

print(f"Edges with cutoff {cutoff}A: {len(edge_src)}")
print(f"\nSample of 5 edges (src, dst, distance):")
for i in range(min(5, len(edge_src))):
    s, d = edge_src[i], edge_dst[i]
    print(f"  {s} -> {d}  dist={dist_matrix[s,d]:.2f}A  "
          f"radii=({accessible_radii[s]:.2f}, {accessible_radii[d]:.2f})")

# Build actual DGL graph
print("\n=== DGL pore graph ===")
try:
    import torch, dgl
    P = dgl.graph((edge_src.tolist(), edge_dst.tolist()))
    P.ndata['h'] = torch.tensor(accessible_radii, dtype=torch.float32).unsqueeze(1)
    edge_dists = dist_matrix[edge_src, edge_dst]
    P.edata['h'] = torch.tensor(edge_dists, dtype=torch.float32).unsqueeze(1)
    print(f"DGL pore graph: {P.num_nodes()} nodes, {P.num_edges()} edges")
    print(f"Node feature shape: {P.ndata['h'].shape}")
    print(f"Edge feature shape: {P.edata['h'].shape}")
    print("\nThis is a valid pore graph ready for GNN message passing.")
except Exception as e:
    print(f"DGL FAILED: {type(e).__name__}: {e}")

for p in [cssr_path, rad_path]:
    if os.path.exists(p): os.unlink(p)

Obtaining hMOF dataset 137k...
Reference:https://doi.org/10.1021/acs.jpcc.6b08729
Loading the zipfile...
Loading completed.
True b'/tmp/tmp4gj4f1q3.rad'
Reading input file: /tmp/tmp4gj4f1q3.cssr
=== analyze_writeto_XYZ ===
Box dimensions:
  va=(19.077100 0 0)
  vb=(0.000000 21.378400 0)
  vc=(0.000000 0.000000 21.377500)

Total particles = 123

Internal grid size = (3 3 3)

Using voro++ with radii for particles.
Performing Voronoi decomposition.
Volume check:
  Total domain volume  = 8718.554165
  Total Voronoi volume = 8718.554165
Voronoi decomposition finished. Rerouting Voronoi network information.
Finished rerouting information.

Finding channels and pockets in Dijkstra network of 746 node(s). 614 are expected to compose pores.
Analyzed and assigned 746 nodes.
Identified 1 channels and 1 pockets.
614 nodes assigned to pores. 
FAILED: FileNotFoundError: [Errno 2] No such file or directory: '/tmp/tmp4gj4f1q3_analyze.xyz'

=== write_to_XYZ with floats ===
Lines: 616
First 25 lines:
61

In [16]:
# Parse pyzeo's own XYZ output (614 nodes, already properly filtered)
lines = open(xyz_path).readlines()
n_nodes = int(lines[0].strip())
nodes = []
for line in lines[2:2+n_nodes]:
    parts = line.split()
    x, y, z, r = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
    nodes.append((x, y, z, r))
print(f"Parsed {len(nodes)} nodes")
print(f"Radius range: {min(n[3] for n in nodes):.2f} — {max(n[3] for n in nodes):.2f} Å")

# Test edge sparsity at different cutoffs
import numpy as np
from scipy.spatial.distance import cdist
positions = np.array([[n[0], n[1], n[2]] for n in nodes])
for cutoff in [3.0, 4.0, 5.0, 6.0]:
    D = cdist(positions, positions)
    src, dst = np.where((D < cutoff) & (D > 0))
    print(f"cutoff={cutoff}Å → {len(src)} edges, avg {len(src)/len(nodes):.1f} edges/node")

Parsed 614 nodes
Radius range: 1.20 — 10.42 Å
cutoff=3.0Å → 9850 edges, avg 16.0 edges/node
cutoff=4.0Å → 16236 edges, avg 26.4 edges/node
cutoff=5.0Å → 22922 edges, avg 37.3 edges/node
cutoff=6.0Å → 32582 edges, avg 53.1 edges/node


In [17]:
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.cluster import DBSCAN

positions = np.array([[n[0], n[1], n[2]] for n in nodes])
radii = np.array([n[3] for n in nodes])

# DBSCAN: group nodes within 3Å of each other into one cluster
clustering = DBSCAN(eps=3.0, min_samples=1).fit(positions)
labels = clustering.labels_
n_clusters = len(set(labels))

print(f"Original nodes: {len(nodes)}")
print(f"Clusters (distinct pores): {n_clusters}")

# Representative node per cluster = the one with largest radius
cluster_positions = []
cluster_radii = []
for c in range(n_clusters):
    mask = labels == c
    idx = np.where(mask)[0]
    best = idx[radii[idx].argmax()]  # node with largest radius in cluster
    cluster_positions.append(positions[best])
    cluster_radii.append(radii[best])

cluster_positions = np.array(cluster_positions)
cluster_radii = np.array(cluster_radii)

print(f"Cluster radius range: {cluster_radii.min():.2f} — {cluster_radii.max():.2f} Å")

# Now test edge density on clustered nodes
for cutoff in [5.0, 8.0, 10.0, 12.0]:
    D = cdist(cluster_positions, cluster_positions)
    src, dst = np.where((D < cutoff) & (D > 0))
    print(f"cutoff={cutoff}Å → {len(src)} edges, avg {len(src)/n_clusters:.1f} edges/node")

Original nodes: 614
Clusters (distinct pores): 11
Cluster radius range: 1.40 — 10.42 Å
cutoff=5.0Å → 8 edges, avg 0.7 edges/node
cutoff=8.0Å → 24 edges, avg 2.2 edges/node
cutoff=10.0Å → 36 edges, avg 3.3 edges/node
cutoff=12.0Å → 46 edges, avg 4.2 edges/node
